In [1]:
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum, avg, col, upper, lower, regexp_replace, when, concat, lit, coalesce, try_to_timestamp, current_timestamp, broadcast, round
from pyspark.sql.types import StructField, StructType, StringType, DoubleType, IntegerType, TimestampType

spark = SparkSession.builder \
        .appName("SparkProject1") \
        .master("local[*]") \
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

raw_schema = StructType([
    StructField("db_id",               StringType(), True),
    StructField("db_name",             StringType(), True),
    StructField("region",              StringType(), True),
    StructField("db_type",             StringType(), True),
    StructField("environment",         StringType(), True),
    StructField("cpu_utilization",     StringType(), True),
    StructField("memory_utilization",  StringType(), True),
    StructField("disk_utilization",    StringType(), True),
    StructField("active_sessions",     StringType(), True),
    StructField("status",              StringType(), True),
    StructField("sla_target",          StringType(), True),
    StructField("recorded_at",         StringType(), True),
])

raw_df = spark.read \
        .option("header", "true") \
        .schema(raw_schema) \
        .csv("/home/ashu/pyspark_learning/project1/data/fleet_metrics_raw.csv")

raw_df.filter(
    (raw_df.active_sessions.cast(IntegerType()) < 0) |
    raw_df.active_sessions.cast(IntegerType()).isNull()
).select("db_id", "db_name", "active_sessions").show(20)

total_row_count = raw_df.count()
print(f"Total count: {total_row_count}")

print("====Duplicates removal====")
raw_df = raw_df.drop_duplicates()
print(f"DF count after dropping duplicates: {raw_df.count()}")

print("====REGION Normalization====")
raw_df = raw_df.withColumn(
    "region",
    regexp_replace(lower(col("region")), "_", "-")
)

raw_df.select(col("region")).distinct().show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 08:45:32 WARN Utils: Your hostname, Ashu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/27 08:45:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 08:45:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----+--------------------+---------------+
|db_id|             db_name|active_sessions|
+-----+--------------------+---------------+
|  187|elasticsearch_pro...|             -6|
|  254|elasticsearch_sta...|            -40|
|  173|    cassandra_dr_083|            -11|
|  226|  cassandra_prod_052|             -6|
|  300|       redis_dev_040|            -39|
|  121| mongodb_staging_031|            -44|
|  200| mongodb_staging_026|            -29|
|  155|   dynamodb_prod_065|             -6|
|  320|  oracle_staging_060|            -47|
|  410|     oracle_prod_057|            -45|
|  339|    cassandra_dr_079|             -1|
+-----+--------------------+---------------+

Total count: 529
====Duplicates removal====
DF count after dropping duplicates: 504
====REGION Normalization====


+--------------+
|        region|
+--------------+
|    ap-south-1|
|     us-west-2|
|     us-east-1|
|     eu-west-1|
|ap-southeast-1|
|  eu-central-1|
+--------------+



In [2]:
raw_df = raw_df.withColumn("status", upper(col("status")))

raw_df = raw_df.withColumn("status",
    when(col("status") == "CRITICAL", "CRITICAL")
    .when(col("status").isin(["WARNING", "WARN"]), "WARNING")
    .when(col("status").isin(["HEALTHY", "OK"]), "HEALTHY")
    .otherwise("UNKNOWN")
)
raw_df.select("status").distinct().show()

+--------+
|  status|
+--------+
| WARNING|
| HEALTHY|
| UNKNOWN|
|CRITICAL|
+--------+



In [3]:
raw_df = raw_df.withColumn(
    "cpu_utilization",
    col("cpu_utilization").cast(DoubleType())
)

raw_df = raw_df.withColumn("active_sessions", col("active_sessions").cast(IntegerType()))
raw_df = raw_df.withColumn(
    "active_sessions",
    when(col("active_sessions") < 0, None)
    .otherwise(col("active_sessions"))
)

raw_df.select(col("cpu_utilization"), col("active_sessions")) \
        .distinct().show()

+---------------+---------------+
|cpu_utilization|active_sessions|
+---------------+---------------+
|          15.99|            198|
|          28.34|            422|
|          89.37|            431|
|          68.33|            476|
|          12.34|            280|
|          48.34|            344|
|          15.98|            446|
|          64.73|            435|
|          22.25|             38|
|          59.68|            354|
|          55.74|             71|
|          33.41|             23|
|          28.81|            182|
|           18.5|            268|
|          57.49|            392|
|          47.49|             47|
|          26.96|            394|
|          12.42|            123|
|          66.91|             37|
|           22.0|            109|
+---------------+---------------+
only showing top 20 rows


In [ ]:
raw_df = raw_df.withColumn(
    "db_name",
    when(col("db_name").isNull() | (col("db_name") == ""), 
        concat(lit("UNKNOWN_DB_"), col("db_id")))
    .otherwise(col("db_name"))
)

raw_df.select("db_name").distinct().show(truncate=False)

+-------------------------+
|db_name                  |
+-------------------------+
|cassandra_dr_055         |
|elasticsearch_staging_080|
|dynamodb_dev_049         |
|cassandra_dr_062         |
|dynamodb_prod_038        |
|oracle_prod_022          |
|oracle_dev_021           |
|cassandra_staging_074    |
|postgres_staging_055     |
|mysql_dr_076             |
|mysql_prod_037           |
|mysql_dr_029             |
|cassandra_dr_051         |
|mongodb_dev_078          |
|elasticsearch_staging_024|
|cassandra_dev_034        |
|redis_staging_023        |
|postgres_prod_032        |
|oracle_staging_074       |
|cassandra_staging_077    |
+-------------------------+
only showing top 20 rows


In [5]:


raw_df = raw_df.withColumn(
    "recorded_at",
    coalesce(
        try_to_timestamp(col("recorded_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("recorded_at"), lit("MM-dd-yyyy HH:mm:ss")),
        try_to_timestamp(col("recorded_at"), lit("MM-dd-yyyy HH:mm")),
        try_to_timestamp(col("recorded_at"), lit("MM/dd/yyyy HH:mm")),
        try_to_timestamp(col("recorded_at"), lit("yyyy/MM/dd HH:mm:ss")),
        try_to_timestamp(col("recorded_at"), lit("dd/MM/yyyy HH:mm")), 
    )
)

# Flag rows where timestamp looks suspicious
# If month > 2 for data that should be Feb 2026 — likely wrong format
from pyspark.sql.functions import month

raw_df = raw_df.withColumn(
    "timestamp_flag",
    when(col("recorded_at").isNull(), "PARSE_FAILED")
    .when(month(col("recorded_at")) > 2, "AMBIGUOUS_FORMAT")
    .otherwise("OK")
)

# Check how many flagged
raw_df.groupBy("timestamp_flag").count().show()

raw_df.select("recorded_at", "timestamp_flag").distinct().show()

+----------------+-----+
|  timestamp_flag|count|
+----------------+-----+
|AMBIGUOUS_FORMAT|   41|
|              OK|  463|
+----------------+-----+

+-------------------+--------------+
|        recorded_at|timestamp_flag|
+-------------------+--------------+
|2026-02-02 09:18:00|            OK|
|2026-02-20 09:23:00|            OK|
|2026-02-04 09:15:00|            OK|
|2026-02-15 07:59:00|            OK|
|2026-02-21 04:58:00|            OK|
|2026-02-14 22:08:00|            OK|
|2026-02-17 17:13:00|            OK|
|2026-02-14 14:56:00|            OK|
|2026-02-04 17:37:00|            OK|
|2026-02-05 15:58:00|            OK|
|2026-02-06 08:49:00|            OK|
|2026-02-25 23:37:00|            OK|
|2026-02-12 04:23:00|            OK|
|2026-02-20 20:06:00|            OK|
|2026-02-15 20:01:00|            OK|
|2026-02-07 10:31:00|            OK|
|2026-02-02 00:19:00|            OK|
|2026-02-14 20:54:00|            OK|
|2026-02-25 07:20:00|            OK|
|2026-02-06 20:31:00|            OK

In [6]:
raw_df = raw_df.withColumns({"ingested_at":current_timestamp(), "pipeline_version":lit("v1.0")})
print(raw_df.columns)

['db_id', 'db_name', 'region', 'db_type', 'environment', 'cpu_utilization', 'memory_utilization', 'disk_utilization', 'active_sessions', 'status', 'sla_target', 'recorded_at', 'timestamp_flag', 'ingested_at', 'pipeline_version']


In [7]:
raw_df = raw_df \
    .withColumn("db_id",              col("db_id").cast(IntegerType())) \
    .withColumn("memory_utilization", col("memory_utilization").cast(DoubleType())) \
    .withColumn("disk_utilization",   col("disk_utilization").cast(DoubleType())) \
    .withColumn("sla_target",         col("sla_target").cast(DoubleType())) \
    .withColumn("cpu_utilization",     col("cpu_utilization").cast(DoubleType())) \
    .withColumn("active_sessions",     col("active_sessions").cast(IntegerType())) \

raw_df.printSchema()
clean_df = raw_df

root
 |-- db_id: integer (nullable = true)
 |-- db_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- db_type: string (nullable = true)
 |-- environment: string (nullable = true)
 |-- cpu_utilization: double (nullable = true)
 |-- memory_utilization: double (nullable = true)
 |-- disk_utilization: double (nullable = true)
 |-- active_sessions: integer (nullable = true)
 |-- status: string (nullable = false)
 |-- sla_target: double (nullable = true)
 |-- recorded_at: timestamp (nullable = true)
 |-- timestamp_flag: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)
 |-- pipeline_version: string (nullable = false)



In [8]:
print("=" * 50)
print("CLEANING SUMMARY REPORT")
print("=" * 50)

cleaned_total = raw_df.count()
print(f"Rows before cleaning : {total_row_count}")
print(f"Rows after cleaning  : {cleaned_total}")
print(f"Rows removed (dupes) : {total_row_count - cleaned_total}")
print()

# Missing value check after cleaning
print("POST-CLEANING NULL CHECK:")
for col_name in ["db_name", "cpu_utilization", "active_sessions",
                 "region", "status", "recorded_at"]:
    null_count = raw_df.filter(col(col_name).isNull()).count()
    flag = " ← EXPECTED (missing cpu)" if col_name == "cpu_utilization" and null_count > 0 else ""
    flag = " ← PROBLEM" if col_name != "cpu_utilization" and null_count > 0 else flag
    print(f"  {col_name:25s}: {null_count} nulls{flag}")

print()

# Verify region normalization worked
print("REGION VARIANTS AFTER CLEANING:")
raw_df.groupBy("region").count().orderBy("region").show()

# Verify status normalization worked
print("STATUS VARIANTS AFTER CLEANING:")
raw_df.groupBy("status").count().orderBy("status").show()

# Verify UNKNOWN_DB replacement
print("UNKNOWN DB ROWS:")
raw_df.filter(col("db_name").startswith("UNKNOWN_DB_")) \
      .select("db_id", "db_name", "region").show()


CLEANING SUMMARY REPORT
Rows before cleaning : 529
Rows after cleaning  : 504
Rows removed (dupes) : 25

POST-CLEANING NULL CHECK:
  db_name                  : 0 nulls
  cpu_utilization          : 22 nulls ← EXPECTED (missing cpu)
  active_sessions          : 11 nulls ← PROBLEM
  region                   : 0 nulls
  status                   : 0 nulls
  recorded_at              : 0 nulls

REGION VARIANTS AFTER CLEANING:
+--------------+-----+
|        region|count|
+--------------+-----+
|    ap-south-1|   74|
|ap-southeast-1|   77|
|  eu-central-1|   93|
|     eu-west-1|   86|
|     us-east-1|   90|
|     us-west-2|   84|
+--------------+-----+

STATUS VARIANTS AFTER CLEANING:
+--------+-----+
|  status|count|
+--------+-----+
|CRITICAL|   46|
| HEALTHY|  336|
| UNKNOWN|   22|
| WARNING|  100|
+--------+-----+

UNKNOWN DB ROWS:
+-----+--------------+--------------+
|db_id|       db_name|        region|
+-----+--------------+--------------+
|  154|UNKNOWN_DB_154|     us-west-2|
|  248|U

In [9]:
print(clean_df.count())
clean_df.show()

504
+-----+--------------------+--------------+-------------+-----------+---------------+------------------+----------------+---------------+-------+----------+-------------------+----------------+--------------------+----------------+
|db_id|             db_name|        region|      db_type|environment|cpu_utilization|memory_utilization|disk_utilization|active_sessions| status|sla_target|        recorded_at|  timestamp_flag|         ingested_at|pipeline_version|
+-----+--------------------+--------------+-------------+-----------+---------------+------------------+----------------+---------------+-------+----------+-------------------+----------------+--------------------+----------------+
|  305|      mysql_prod_045|  eu-central-1|        mysql|       prod|          84.71|             42.67|           33.39|            459|WARNING|      99.0|2026-02-13 07:48:00|              OK|2026-02-27 08:46:...|            v1.0|
|  375|    cassandra_dr_022|    ap-south-1|    cassandra|         dr

In [10]:
region_config = spark.read \
                .option("header", "true") \
                .option("inferschema", "true") \
                .csv("/home/ashu/pyspark_learning/project1/data/region_config.csv")

enriched_df = clean_df.join(broadcast(region_config), on="region", how="left")
enriched_df.cache()
enriched_df.count()
enriched_df.printSchema()

root
 |-- region: string (nullable = true)
 |-- db_id: integer (nullable = true)
 |-- db_name: string (nullable = true)
 |-- db_type: string (nullable = true)
 |-- environment: string (nullable = true)
 |-- cpu_utilization: double (nullable = true)
 |-- memory_utilization: double (nullable = true)
 |-- disk_utilization: double (nullable = true)
 |-- active_sessions: integer (nullable = true)
 |-- status: string (nullable = false)
 |-- sla_target: double (nullable = true)
 |-- recorded_at: timestamp (nullable = true)
 |-- timestamp_flag: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)
 |-- pipeline_version: string (nullable = false)
 |-- team_name: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- alert_email: string (nullable = true)



In [11]:
risk_score_df = enriched_df.withColumn("cpu_weight", round(coalesce(col("cpu_utilization"), lit(0))*0.4, 2)) \
                .withColumn("memory_weight", round(col("memory_utilization")*0.3, 2)) \
                .withColumn("disk_weight", round(col("disk_utilization")*0.2, 2)) \
                .withColumn("session_weight",round((coalesce(col("active_sessions"), lit(0)) / lit(500.0)) * lit(100.0) * lit(0.1),2)) \
                .withColumn("raw_score", round(col("cpu_weight") + col("memory_weight") + col("disk_weight") + col("session_weight"), 2))
                

risk_score_df = risk_score_df.withColumn(
    "sla_multiplier",
    when(col("sla_target") >=  99.9, 1.5) \
    .when(col("sla_target") >= 99.5, 1.2) \
    .when(col("sla_target") >= 99.0, 1.0) \
)

risk_score_df = risk_score_df.withColumn(
    "final_risk_score",
    round(col("raw_score")*col("sla_multiplier"), 2)
)

risk_score_df = risk_score_df.withColumn(
    "risk_category",
    when(col("final_risk_score")>=80, "CRITICAL_RISK") \
    .when(col("final_risk_score")>=60, "HIGH_RISK") \
    .when(col("final_risk_score")>=40, "MEDIUM_RISK") \
    .otherwise("LOW_RISK")
)

# Verify max possible is ~150
print("Max risk score:", risk_score_df.agg({"final_risk_score": "max"}).collect()[0][0])
print("Min risk score:", risk_score_df.agg({"final_risk_score": "min"}).collect()[0][0])
print("Avg risk score:", risk_score_df.agg({"final_risk_score": "avg"}).collect()[0][0])

# Null check
print("Null risk scores:", risk_score_df.filter(col("final_risk_score").isNull()).count())

# Distribution
risk_score_df.groupBy("risk_category").count().orderBy("risk_category").show()
# risk_score_df.printSchema()
risk_score_df.show()


Max risk score: 127.55
Min risk score: 13.24
Avg risk score: 63.20690476190476
Null risk scores: 0


+-------------+-----+
|risk_category|count|
+-------------+-----+
|CRITICAL_RISK|  115|
|    HIGH_RISK|  155|
|     LOW_RISK|   72|
|  MEDIUM_RISK|  162|
+-------------+-----+

+--------------+-----+--------------------+-------------+-----------+---------------+------------------+----------------+---------------+-------+----------+-------------------+----------------+--------------------+----------------+-------------+--------+--------------------+----------+-------------+-----------+--------------+---------+--------------+----------------+-------------+
|        region|db_id|             db_name|      db_type|environment|cpu_utilization|memory_utilization|disk_utilization|active_sessions| status|sla_target|        recorded_at|  timestamp_flag|         ingested_at|pipeline_version|    team_name|timezone|         alert_email|cpu_weight|memory_weight|disk_weight|session_weight|raw_score|sla_multiplier|final_risk_score|risk_category|
+--------------+-----+--------------------+------------

In [12]:
risk_score_df.cache()
risk_score_df.count()
regional_benchmark = risk_score_df.groupBy("region", "team_name", "timezone") \
                    .agg( count(col("db_name")).alias("total_db"),
                        round(avg(col("cpu_utilization")), 2).alias("avg_cpu"), 
                        round(avg(col("memory_utilization")), 2).alias("avg_memory"), 
                        round(avg(col("disk_utilization")), 2).alias("avg_disk"),
                        sum(when(col("status")=="HEALTHY", 1).otherwise(0)).alias("healthy_count"),
                        sum(when(col("status")=="WARNING", 1).otherwise(0)).alias("warning_count"),
                        sum(when(col("status")=="CRITICAL", 1).otherwise(0)).alias("critical_count"),
                        round(avg(col("final_risk_score")), 2).alias("avg_risk_score"),
                        round(
                            sum(when(col("status") == "HEALTHY", 1).otherwise(0)) /
                            count("db_name") * 100, 2
                            ).alias("health_percentage")   # ← directly here
                    ) \
                    .orderBy(col("avg_risk_score").desc()) 

print(len(regional_benchmark.columns))
regional_benchmark.show()

12
+--------------+-------------+--------+--------+-------+----------+--------+-------------+-------------+--------------+--------------+-----------------+
|        region|    team_name|timezone|total_db|avg_cpu|avg_memory|avg_disk|healthy_count|warning_count|critical_count|avg_risk_score|health_percentage|
+--------------+-------------+--------+--------+-------+----------+--------+-------------+-------------+--------------+--------------+-----------------+
|     us-west-2|Team_Americas|   UTC-8|      84|  52.83|     59.75|   49.47|           57|           13|            13|          65.3|            67.86|
|    ap-south-1|    Team_APAC|   UTC+5|      74|  55.92|     51.28|   50.25|           48|           18|             6|          64.4|            64.86|
|  eu-central-1|  Team_Europe|   UTC+2|      93|  52.08|      56.1|   50.28|           66|           20|             5|         64.34|            70.97|
|ap-southeast-1|    Team_APAC|   UTC+8|      77|  53.67|     55.33|   45.75|   

In [13]:
incident_classification_df = risk_score_df.withColumn(
    "incident_classification",
    when((col("cpu_utilization") > 90) & (col("active_sessions") > 300), "IMMEDIATE_PAGE")
    .when((col("cpu_utilization") > 80) & (col("active_sessions") > 200), "URGENT_TICKET")
    .when(col("memory_utilization")  > 85, "MEMORY_ALERT")
    .when(col("disk_utilization") > 80, "DISK_ALERT")
    .otherwise("MONITOR_CLOSELY")
)

incident_classification_df.filter(col("risk_category").isin(["CRITICAL_RISK", "HIGH_RISK"])) \
                            .select("db_name", "region", "cpu_utilization", "memory_utilization", "disk_utilization", "team_name", "alert_email", "risk_category", "incident_classification", "final_risk_score") \
                            .orderBy(col("final_risk_score").desc()).show()

+--------------------+--------------+---------------+------------------+----------------+-------------+--------------------+-------------+-----------------------+----------------+
|             db_name|        region|cpu_utilization|memory_utilization|disk_utilization|    team_name|         alert_email|risk_category|incident_classification|final_risk_score|
+--------------------+--------------+---------------+------------------+----------------+-------------+--------------------+-------------+-----------------------+----------------+
|        mysql_dr_076|ap-southeast-1|           80.1|             94.71|           73.02|    Team_APAC|apac-oncall@compa...|CRITICAL_RISK|          URGENT_TICKET|          127.55|
|elasticsearch_dr_050|     eu-west-1|          95.77|             70.46|           89.27|  Team_Europe|europe-oncall@com...|CRITICAL_RISK|          URGENT_TICKET|          124.74|
|    cassandra_dr_024|     us-east-1|          96.54|             75.95|           88.32|Team_Americ

In [14]:
risk_score_df = risk_score_df.withColumn("cpu_flag", when(col("cpu_utilization")>70, "REVIEW_NEEDED").otherwise("OK")) \
                                .withColumn("memory_flag", when(col("memory_utilization")>75, "REVIEW_NEEDED").otherwise("OK")) \
                                .withColumn("disk_flag", when(col("disk_utilization")>70, "REVIEW_NEEDED").otherwise("OK"))

risk_score_df.filter((col("cpu_flag")=="REVIEW_NEEDED") | (col("memory_flag")=="REVIEW_NEEDED") | (col("disk_flag")=="REVIEW_NEEDED")) \
                .select("db_name", "region", "cpu_utilization", "memory_utilization", "disk_utilization", "team_name", "alert_email", "cpu_flag", "memory_flag", "disk_flag") \
                .orderBy("region") \
                .show()

+--------------------+----------+---------------+------------------+----------------+---------+--------------------+-------------+-------------+-------------+
|             db_name|    region|cpu_utilization|memory_utilization|disk_utilization|team_name|         alert_email|     cpu_flag|  memory_flag|    disk_flag|
+--------------------+----------+---------------+------------------+----------------+---------+--------------------+-------------+-------------+-------------+
|      redis_prod_048|ap-south-1|           NULL|              16.2|           70.48|Team_APAC|apac-oncall@compa...|           OK|           OK|REVIEW_NEEDED|
|      mongodb_dr_032|ap-south-1|          71.01|              44.7|           33.63|Team_APAC|apac-oncall@compa...|REVIEW_NEEDED|           OK|           OK|
|    mongodb_prod_069|ap-south-1|          63.32|             39.54|            88.9|Team_APAC|apac-oncall@compa...|           OK|           OK|REVIEW_NEEDED|
|      oracle_dev_029|ap-south-1|          94.

In [15]:
enriched_df.unpersist()
risk_score_df.unpersist()

DataFrame[region: string, db_id: int, db_name: string, db_type: string, environment: string, cpu_utilization: double, memory_utilization: double, disk_utilization: double, active_sessions: int, status: string, sla_target: double, recorded_at: timestamp, timestamp_flag: string, ingested_at: timestamp, pipeline_version: string, team_name: string, timezone: string, alert_email: string, cpu_weight: double, memory_weight: double, disk_weight: double, session_weight: double, raw_score: double, sla_multiplier: double, final_risk_score: double, risk_category: string, cpu_flag: string, memory_flag: string, disk_flag: string]

In [17]:
clean_df.write.mode('overwrite').partitionBy("region", "environment").parquet("/home/ashu/pyspark_learning/project1/output_data/clean_metrics")

csv_file_path = f"/home/ashu/pyspark_learning/project1/incidents/incident_data_{datetime.now().strftime('%Y_%m_%d')}.csv"
incident_classification_df.filter(col("risk_category").isin(["CRITICAL_RISK", "HIGH_RISK"])) \
                            .coalesce(1) \
                            .write \
                            .mode('overwrite') \
                            .option("header", "true") \
                            .csv(csv_file_path)

regional_benchmark.coalesce(1) \
                    .write \
                    .mode('overwrite') \
                    .parquet("/home/ashu/pyspark_learning/project1/output_data/regional_summary")

In [ ]:
df_parquet = spark.read \
    .parquet("/home/ashu/pyspark_learning/project1/output_data/clean_metrics")

df_parquet.filter(col("region") == "us-east-1").explain(True)

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [db_id#8918,db_name#8919,db_type#8920,cpu_utilization#8921,memory_utilization#8922,disk_utilization#8923,active_sessions#8924,status#8925,sla_target#8926,recorded_at#8927,timestamp_flag#8928,ingested_at#8929,pipeline_version#8930,region#8931,environment#8932] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/ashu/pyspark_learning/project1/output_data/clean_metrics], PartitionFilters: [isnotnull(region#8931), (region#8931 = us-east-1)], PushedFilters: [], ReadSchema: struct<db_id:int,db_name:string,db_type:string,cpu_utilization:double,memory_utilization:double,d...




In [21]:
# Verify all three outputs exist
import subprocess

paths = [
    "/home/ashu/pyspark_learning/project1/output_data/clean_metrics",
    "/home/ashu/pyspark_learning/project1/incidents",
    "/home/ashu/pyspark_learning/project1/output_data/regional_summary"
]

for path in paths:
    result = subprocess.run(
        ["find", path, "-type", "f", "-name", "*.parquet", "-o",
         "-type", "f", "-name", "*.csv"],
        capture_output=True, text=True
    )
    files = [f for f in result.stdout.strip().split("\n") if f]
    print(f"{path.split('/')[-1]}: {len(files)} files")

clean_metrics: 24 files
incidents: 1 files
regional_summary: 1 files
